# 14.05 · Multimodal RAG

**Chapter:** Multimodal AI · **Milestone:** M7

## Goals
- [ ] See why text-only RAG misses information that lives in charts, tables-as-images, scans and slides
- [ ] Compare four page retrievers on the same eval: **extracted text**, **CLIP** image embeddings, **VLM captions**, and **ColPali**-style late interaction
- [ ] Implement late-interaction (MaxSim) scoring yourself
- [ ] Answer from the retrieved page **image** with a VLM, with page citations

## Resources
- [ColPali paper](https://arxiv.org/abs/2407.01449) · [colpali-engine](https://github.com/illuin-tech/colpali) · [ViDoRe benchmark](https://huggingface.co/spaces/vidore/vidore-leaderboard) — free
- [CLIP](https://arxiv.org/abs/2103.00020) · [SigLIP 2](https://arxiv.org/abs/2502.14786) — free
- [HF – Multimodal RAG with ColQwen2 + Qwen2-VL](https://huggingface.co/learn/cookbook/multimodal_rag_using_document_retrieval_and_vlms) — free
- [Byaldi](https://github.com/AnswerDotAI/byaldi) (ColPali in a few lines) — free

Real document collections are full of content text extraction can't see: a chart pasted as a picture in a slide deck, a scanned contract,
a table exported as an image. Multimodal RAG retrieves **pages as images** (or their visual descriptions) and lets a VLM read them.

In [ ]:
import io
import json
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pymupdf
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image

from aieng import DATA, rag
from aieng.llm import available, client

WORK = DATA / "mmrag"
PAGES = WORK / "pages"
PAGES.mkdir(parents=True, exist_ok=True)

## 1. A mixed corpus: a research paper + a slide deck whose numbers live in images
The deck is synthetic (so we know every answer). Slide titles are real text, but each chart is pasted as a **raster image**, the way
screenshots end up in real decks. The paper is "Attention Is All You Need" (15 text-heavy pages, from 08.02).

In [ ]:
def chart_png(draw, figsize=(7, 3.6)):
    fig, ax = plt.subplots(figsize=figsize)
    draw(ax)
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    return Image.open(buf)


def bars(labels, values, fmt="{}", horizontal=False, color="#4c78a8"):
    def draw(ax):
        (ax.barh if horizontal else ax.bar)(labels, values, color=color)
        for i, v in enumerate(values):
            ax.text(v, i, " " + fmt.format(v), va="center") if horizontal else ax.text(i, v, fmt.format(v), ha="center", va="bottom")
        ax.spines[["top", "right"]].set_visible(False)
    return draw


def table(rows):
    def draw(ax):
        ax.axis("off"); t = ax.table(cellText=rows[1:], colLabels=rows[0], loc="center", cellLoc="center"); t.scale(1, 2); t.set_fontsize(12)
    return draw


def timeline(events):
    def draw(ax):
        ax.axhline(0, color="gray"); ax.axis("off"); ax.set_ylim(-1, 1)
        for i, (when, what) in enumerate(events):
            ax.plot(i, 0, "o", ms=14, color="#e45756"); ax.text(i, 0.25, when, ha="center", fontsize=13, weight="bold")
            ax.text(i, -0.35, what, ha="center", fontsize=11)
        ax.set_xlim(-0.6, len(events) - 0.4)
    return draw


SLIDES = [  # (title, chart or None, bullet text or None)
    ("Andes Coffee Co. — Q3 2026 Board Review", None, "Confidential. Prepared by the finance team."),
    ("Revenue by quarter", bars(["Q4 2025", "Q1 2026", "Q2 2026", "Q3 2026"], [1.8, 2.1, 2.4, 3.1], "${}M"), None),
    ("Revenue mix by channel", lambda ax: ax.pie([45, 35, 20], labels=["Cafés", "Online", "Wholesale"], autopct="%d%%", startangle=90), None),
    ("Customer satisfaction", lambda ax: (ax.plot(["Jul", "Aug", "Sep"], [41, 47, 52], "o-", lw=3),
                                          [ax.text(i, v + 0.6, f"NPS {v}", ha="center") for i, v in enumerate([41, 47, 52])]), None),
    ("Store openings this quarter", table([["City", "New stores"], ["Lima", "3"], ["Cusco", "2"], ["Arequipa", "1"]]), None),
    ("Unit economics", bars(["Beans", "Milk", "Labor", "Rent"], [0.42, 0.18, 0.65, 0.30], "${:.2f}", horizontal=True, color="#f58518"), None),
    ("Best sellers", bars(["Flat white", "Latte", "Cold brew", "Mocha"], [31, 28, 19, 12], "{}k cups", color="#54a24b"), None),
    ("Roadmap", timeline([("October", "Mobile app"), ("November", "Loyalty program"), ("December", "Santiago store")]), None),
    ("Hiring plan", bars(["Baristas", "Roasters", "Engineers", "Marketing"], [40, 6, 4, 2], "{}", color="#b279a2"), None),
    ("Risks", None, "• Coffee bean prices up 18% year over year\n• PEN/USD currency volatility\n• Lease renewals in Miraflores"),
]

with PdfPages(WORK / "deck.pdf") as pdf:
    for title, chart, text in SLIDES:
        fig = plt.figure(figsize=(12.8, 7.2))
        fig.text(0.05, 0.9, title, fontsize=28, weight="bold")  # real (extractable) text
        if chart:
            ax = fig.add_axes([0.1, 0.05, 0.8, 0.75]); ax.imshow(chart_png(chart)); ax.axis("off")  # chart as an image
        if text:
            fig.text(0.07, 0.65, text, fontsize=22, va="top")
        pdf.savefig(fig); plt.close(fig)

# Every page of every PDF -> one PNG + its extractable text
pages = []
for doc_name in ("deck.pdf", "attention.pdf"):
    path = WORK / doc_name if doc_name == "deck.pdf" else DATA / doc_name
    for i, page in enumerate(pymupdf.open(path)):
        name = f"{doc_name.split('.')[0]}-p{i + 1:02d}"
        page.get_pixmap(dpi=72).save(PAGES / f"{name}.png")
        pages.append({"name": name, "text": page.get_text()})
pages.sort(key=lambda p: p["name"])  # same order as sorted(glob) in colpali_embed.py
print(len(pages), "pages")
print("text extracted from the revenue slide:", repr(next(p for p in pages if p["name"] == "deck-p02")["text"]))
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, n in zip(axes, ["deck-p02", "deck-p05", "deck-p08", "attention-p05"]):
    ax.imshow(Image.open(PAGES / f"{n}.png")); ax.set_title(n); ax.axis("off")
plt.show()

The text layer of the revenue slide is just its title: the numbers exist only as pixels. Now the eval set — questions, gold page(s) and
acceptable answer strings to check generation later.

In [ ]:
QA = [
    ("What was revenue in Q3 2026?", ["deck-p02"], "3.1"),
    ("What share of revenue came from online sales?", ["deck-p03"], "35"),
    ("What was the NPS in September?", ["deck-p04"], "52"),
    ("How many new stores opened in Cusco?", ["deck-p05"], ("2", "two")),
    ("What is the largest cost per cup?", ["deck-p06"], "labor"),
    ("Which drink sold the most cups?", ["deck-p07"], "flat white"),
    ("When does the loyalty program launch?", ["deck-p08"], "november"),
    ("How many engineers are we hiring?", ["deck-p09"], ("4", "four")),
    ("How much have coffee bean prices increased?", ["deck-p10"], "18"),
    ("How many parallel attention heads does the base Transformer use?", ["attention-p05"], ("8", "eight")),
    ("Which optimizer was used to train the Transformer?", ["attention-p07"], "adam"),
    ("What value of label smoothing was used during training?", ["attention-p08"], "0.1"),
    ("How many sentence pairs does the WMT 2014 English-German dataset have?", ["attention-p07"], "4.5 million"),
    ("What BLEU score does the big Transformer reach on English-to-German?", ["attention-p01", "attention-p08"], "28.4"),
]
names = [p["name"] for p in pages]


def evaluate(name, scores):
    """scores: (n_questions, n_pages). Recall@k = gold page in top k."""
    rows = []
    for (q, gold, _), s in zip(QA, scores):
        rank = [names[i] for i in np.argsort(-s)]
        rows.append((gold[0].startswith("deck"), any(g in rank[:1] for g in gold), any(g in rank[:3] for g in gold)))
    r = np.array(rows)
    deck, paper = r[r[:, 0] == 1], r[r[:, 0] == 0]
    print(f"{name:20} R@1 {r[:, 1].mean():4.0%}  R@3 {r[:, 2].mean():4.0%} | slides R@3 {deck[:, 2].mean():4.0%} | paper R@3 {paper[:, 2].mean():4.0%}")
    return r


questions = [q for q, *_ in QA]
results = {}

## 2. Retriever A — extracted text + text embeddings (classic RAG)

In [ ]:
text_emb = rag.embed([p["text"] or p["name"] for p in pages])
results["text"] = evaluate("A: extracted text", rag.embed(questions, query=True) @ text_emb.T)

## 3. Retriever B — CLIP image embeddings
CLIP embeds images and text in one space (14.04). It's great for photos ("a red bicycle"), but it was trained on captions, not on reading
documents.

In [ ]:
from sentence_transformers import SentenceTransformer

clip = SentenceTransformer("clip-ViT-B-32")
img_emb = clip.encode([Image.open(PAGES / f"{n}.png") for n in names], normalize_embeddings=True)
results["clip"] = evaluate("B: CLIP image", clip.encode(questions, normalize_embeddings=True) @ img_emb.T)

## 4. Retriever C — VLM captions, then text retrieval
Index time: ask a VLM to transcribe each page *including what's in the charts*, append the extracted text, and embed. Query time stays cheap
(plain text retrieval). Cost moves to indexing: one VLM call per page (here local `qwen3-vl:2b-instruct` from 14.01).

In [ ]:
import base64

vc, _ = client("ollama")
CAPTION = "Transcribe this page for a search index: the title, every number, label and value in charts or tables, and the main points. Plain text."
captions_path = WORK / "captions.json"
captions = json.loads(captions_path.read_text()) if captions_path.exists() else {}
start = time.perf_counter()
for n in names:
    if n not in captions:
        b64 = base64.b64encode((PAGES / f"{n}.png").read_bytes()).decode()
        r = vc.chat.completions.create(model="qwen3-vl:2b-instruct", temperature=0, max_tokens=400, messages=[{"role": "user", "content": [
            {"type": "text", "text": CAPTION}, {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}])
        captions[n] = r.choices[0].message.content
captions_path.write_text(json.dumps(captions, indent=1))
print(f"captioned in {time.perf_counter() - start:.0f}s (cached afterwards)\n--- deck-p02 caption:\n{captions['deck-p02'][:300]}")
cap_emb = rag.embed([captions[n] + "\n" + p["text"] for n, p in zip(names, pages)])
results["caption"] = evaluate("C: VLM caption", rag.embed(questions, query=True) @ cap_emb.T)

## 5. Retriever D — ColPali-style late interaction
ColPali (here **ColSmol-256M**) feeds the page image through a VLM and keeps **one 128-d vector per image patch** (~1k per page), and one per
query token. Score = **MaxSim**: for each query token take its best-matching patch, then sum. Fine-grained matching ("Cusco" ↔ the patch
where "Cusco" is written) without any OCR. Same idea as ColBERT for text (07.x).

`colpali-engine` pins older torch/peft, so `colpali_embed.py` runs in an **isolated** env (as Inspect did in 10.05) and writes vectors to disk.

In [ ]:
(WORK / "queries.json").write_text(json.dumps(questions))
r = subprocess.run(["uv", "run", "--no-project", "--with", "colpali-engine", "--with", "numpy", "python", "colpali_embed.py",
                    str(PAGES), str(WORK / "queries.json"), str(WORK / "colsmol.npz")], capture_output=True, text=True,
                   env={**__import__("os").environ, "HF_HUB_DISABLE_XET": "1"})
print(r.stdout.strip() or r.stderr[-1500:])
E = np.load(WORK / "colsmol.npz")
assert list(E["names"]) == [n + ".png" for n in names]

In [ ]:
def maxsim(q, q_len, pages, page_lens):
    q = q[:q_len]
    out = []
    for p, n in zip(pages, page_lens):
        sims = q @ p[:n].T                  # (query tokens, page patches)
        out.append(sims.max(axis=1).sum())  # best patch per query token, summed
    return np.array(out)


col_scores = np.stack([maxsim(q, ql, E["pages"], E["page_lens"]) for q, ql in zip(E["queries"], E["query_lens"])])
results["colpali"] = evaluate("D: ColSmol MaxSim", col_scores)
print(f"index size: {E['page_lens'].sum():,} vectors × 128 for {len(names)} pages (vs {len(names)} vectors for A/B/C)")

Read the table by question type:
- **Extracted text** is strong on the paper and finds slides only through their title words ("hiring", "roadmap").
- **CLIP** can read big, clean words (28 pt slide titles) but not 10 pt body text: great on slides, useless on the paper.
- **Captions** see inside the charts, but appending a caption to a long page dilutes that page's single vector — chunk long pages (08.02).
- **ColPali** handles both without any OCR, at the price of ~1,000 vectors per page.

Trade-offs:

| | Index cost | Query cost | Storage | Sees images | Notes |
|---|---|---|---|---|---|
| Extracted text | cheap | cheap | 1 vector/chunk | ✗ | still the best for clean text-heavy documents |
| CLIP / SigLIP | cheap | cheap | 1 vector/page | weakly | photos, product images |
| VLM captions | 1 VLM call/page | cheap | 1 vector/chunk | ✓ | captions are inspectable text; quality = VLM quality |
| ColPali family | 1 VLM forward/page | VLM encode + MaxSim | ~1k vectors/page | ✓ | best on visually rich docs (ViDoRe); needs multi-vector DB support (Vespa, Qdrant, LanceDB) |

Production systems often combine them: hybrid text + visual retrieval, then rerank.

## 6. Generate: answer from the page image
Retrieve with the best retriever, send the top page **image** to a VLM with the question, cite the page.

In [ ]:
best = max(results, key=lambda k: (results[k][:, 2].mean(), results[k][:, 1].mean()))  # R@3, ties broken by R@1
scores = {"text": rag.embed(questions, query=True) @ text_emb.T, "clip": clip.encode(questions, normalize_embeddings=True) @ img_emb.T,
          "caption": rag.embed(questions, query=True) @ cap_emb.T, "colpali": col_scores}[best]
provider = "gemini" if "gemini" in available() else "ollama"
gc, gmodel = client(provider)
gmodel = gmodel if provider == "gemini" else "qwen3-vl:2b-instruct"
print(f"retriever: {best} | generator: {provider}/{gmodel}\n")
correct = 0
for (q, gold, ans), s in zip(QA, scores):
    top = [names[i] for i in np.argsort(-s)[:2]]
    content = [{"type": "text", "text": f"Answer from these pages only, in one short sentence, and cite the page name in brackets. Pages: {top}. "
                                        f"If they don't contain the answer, say so.\nQuestion: {q}"}]
    content += [{"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode((PAGES / f"{n}.png").read_bytes()).decode()}}
                for n in top]
    out = gc.chat.completions.create(model=gmodel, temperature=0, max_tokens=200, messages=[{"role": "user", "content": content}]).choices[0].message.content
    ok = any(a in out.lower() for a in ((ans,) if isinstance(ans, str) else ans))
    correct += ok
    print(f"{'✅' if ok else '❌'} {q[:55]:55} -> {out.strip()[:90]}")
    if provider == "gemini":
        time.sleep(4)  # free-tier rate limit
print(f"\n{correct}/{len(QA)} answered correctly")

## Exercises
1. RAG over **your** slide decks: export a real deck to PDF, write 15 questions whose answers are in charts/screenshots, and compare A–D.
2. Add hybrid retrieval: combine text and ColPali scores (normalise each, then weighted sum or reciprocal rank fusion from 07.x). Does it beat both?
3. Store ColSmol multi-vectors in LanceDB or Qdrant (both support multi-vector MaxSim) instead of numpy, and time a query over 1,000 pages.
4. Swap the generator to the local VLM and measure how much answer accuracy drops vs Gemini when retrieval is held fixed.

In [ ]:
# your code here